# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini3206/flyrank-ml-workspace/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Why Random Forest & Simple Baseline Fits Lane 2
For Lane 2 (Refresh / Content Opportunity Scoring), our objective is to predict whether a highly visible page's search impressions will decline in the upcoming month. This yes/no classification problem uses probabilities to rank content items. We choose **Random Forest Classifier** as our primary machine learning method, alongside a **Decision Tree** and a **Logistic Regression** for the following reasons:

1. **Non-Linear Relationships:** Tree-based models can easily handle non-linear interactions (e.g., highly visible content that is decaying despite good keyword metrics vs. stale pages with low search demand).
2. **Continuous Probabilities:** Random Forest provides continuous probability estimates, which let us build a continuous, prioritized queue of pages to refresh. This is far more useful for an editorial team with limited weekly bandwidth than a rigid yes/no rule.
3. **Robust Feature Importance:** Out-of-bag metrics and permutation-like feature splits give us an honest understanding of which features (impressions, average position, age) are driving search decline, preventing leakage from being silently incorporated.

In [1]:
# Verification of basic packages and environment configuration
import os
import numpy as np
import pandas as pd
import sklearn
print(f"Using Scikit-Learn version: {sklearn.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


Using Scikit-Learn version: 1.6.1
Numpy version: 2.0.2
Pandas version: 2.2.2


## 2. Split design

### Split Strategy: Temporal Window Alignment with Stratified Row Holdout
To build an honest model, we use the same split design and data contract established in **Week 3 (`w03_data_contract.ipynb`)** and **Week 4 (`w04_baseline_score.ipynb`)**:

- **Feature / Decision Window:** March 2026 (`month = '2026-03'`). All input features are aggregated or computed over this month. This ensures we use only features available at the decision point (March 31, 2026).
- **Label / Outcome Window:** April 2026 (`month = '2026-04'`). The outcome label `is_declining` is evaluated strictly based on whether April search impressions fall below 80% of March search impressions.
- **Stratified Row Holdout (75% Train / 25% Test):** We use a random seed (`random_state=42`) and stratify by the target variable `is_declining` to guarantee that both train and validation splits have identical base rates. This keeps our metric calculations (Precision@50, Average Precision, ROC AUC) highly stable and honest.

In [5]:
# Split verification lives here conceptually; the actual client-grouped split
# and its verification happen in Section 3, against the real dataset.
print("Split design: client-grouped holdout (GroupShuffleSplit on client_hash_id, test_size=0.25).")
print("This prevents a client's pages from appearing in both train and test — see Section 3 for the real numbers.")

Split design: client-grouped holdout (GroupShuffleSplit on client_hash_id, test_size=0.25).
This prevents a client's pages from appearing in both train and test — see Section 3 for the real numbers.


- **Client-Grouped Holdout (75% Train / 25% Test):** We use `GroupShuffleSplit` grouped by
  `client_hash_id` (`random_state=42`) so that no single client's pages appear in both train
  and test — preventing client-level leakage. This matches the client-holdout discipline
  established in Week 2 (`w01`/notebook 02's "Your turn" experiment).

## 3. Train + compare vs my baseline

### Honest Comparison Table: Machine Learning vs. Baseline Rules

Client-grouped holdout: 93,085 train rows / 8,356 test rows, 0 overlapping clients.

| Model | ROC AUC | Average Precision (AP) | Precision@20 | Precision@50 |
|---|---:|---:|---:|---:|
| **Baseline Rules** | 0.5476 | 0.5319 | **1.0000** | **0.9800** |
| Logistic Regression | 0.5757 | 0.5970 | 0.7000 | 0.7800 |
| Random Forest | 0.5799 | 0.5356 | 0.5500 | 0.6200 |
| Decision Tree | 0.5414 | 0.5124 | 0.3500 | 0.4400 |

### Observations
- **The baseline wins decisively at the top of the list, once client leakage is removed.**
  On a true client-grouped split, the hand-written rule (stale AND visible, ranked by
  impressions) achieves Precision@20 = 1.00 and Precision@50 = 0.98 — every one of its
  top 50 picks is genuinely declining. All three ML models score lower at both cutoffs.
- **But the baseline's overall ROC AUC (0.5476) is still the weakest of the four** — it's
  barely better than chance across the *whole* ranking, and only reliable at the very top.
  The rule doesn't generalize a ranking order across all 8,356 test pages; it just happens
  to place a small number of unambiguous "big, old, currently-getting-lots-of-impressions"
  pages first, and those specific pages are genuinely the ones declining.
- **This is a different story than the earlier (non-client-grouped) run**, where Random
  Forest looked dramatically better than the baseline. That gap was partly an artifact of
  client-level leakage — the model was learning client-specific patterns that don't hold up
  once no client's pages appear in both train and test. Honest evaluation: **the baseline is
  hard to beat at the top of the queue on this data slice**, and the ML models' real value —
  if any — would need to be demonstrated further down the ranking (deeper K), not by
  headline top-20/top-50 numbers.

In [6]:
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Setup Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Loading dataset from the warehouse (mid-panel month partitions)...")
dataset = con.execute(f"""
    WITH mar_perf AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS mar_impressions,
            SUM(gsc_clicks) AS mar_clicks,
            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions)
                ELSE NULL
            END AS mar_avg_position,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS mar_ga4_sessions,
            COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS mar_ga4_days
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
    ),
    apr_perf AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS apr_impressions
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.mar_impressions,
        m.mar_clicks,
        m.mar_avg_position,
        CASE WHEN m.mar_ga4_days > 0 THEN m.mar_ga4_sessions / m.mar_ga4_days ELSE 0.0 END AS mar_ga4_sessions_per_day,
        DATEDIFF('day', CAST(c.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) AS content_age_days,
        c.word_count,
        COALESCE(a.apr_impressions, 0) AS apr_impressions,
        CASE
            WHEN a.apr_impressions IS NULL THEN 1
            WHEN a.apr_impressions < 0.8 * m.mar_impressions THEN 1
            ELSE 0
        END AS is_declining
    FROM mar_perf m
    LEFT JOIN apr_perf a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
    LEFT JOIN {TABLES['dim_content']} c
        ON m.content_hash_id = c.content_hash_id
    WHERE m.mar_impressions >= 100
""").df()

dataset = dataset.dropna(subset=['mar_avg_position', 'content_age_days'])
print(f"Loaded clean dataset of {len(dataset):,} content items.")

# Calculate the Baseline Action Score (Week 4)
stale = (dataset["content_age_days"] >= 365).astype(int)
visible = (dataset["mar_impressions"] >= 500).astype(int)
dataset["baseline_score"] = stale * visible * dataset["mar_impressions"]

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

X_cols = ['mar_impressions', 'mar_clicks', 'mar_avg_position', 'mar_ga4_sessions_per_day', 'content_age_days']
y = dataset['is_declining']

from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(dataset, y, groups=dataset["client_hash_id"]))
X_tr, X_te = dataset.iloc[tr_idx], dataset.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

# Verify no client overlap between train and test
overlap = set(X_tr["client_hash_id"]) & set(X_te["client_hash_id"])
print(f"Train: {len(X_tr):,} rows | Test: {len(X_te):,} rows | Overlapping clients: {len(overlap)}")

# Evaluate baseline
base_roc = roc_auc_score(y_te, X_te['baseline_score'])
base_ap = average_precision_score(y_te, X_te['baseline_score'])
base_p20 = precision_at_k(y_te, X_te['baseline_score'], 20)
base_p50 = precision_at_k(y_te, X_te['baseline_score'], 50)
print(f"Baseline: ROC AUC = {base_roc:.4f}, AP = {base_ap:.4f}, P@20 = {base_p20:.4f}, P@50 = {base_p50:.4f}")

# Train & Evaluate Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr[X_cols], y_tr)
lr_probs = lr.predict_proba(X_te[X_cols])[:, 1]
lr_roc = roc_auc_score(y_te, lr_probs)
lr_ap = average_precision_score(y_te, lr_probs)
lr_p20 = precision_at_k(y_te, lr_probs, 20)
lr_p50 = precision_at_k(y_te, lr_probs, 50)
print(f"Logistic Regression: ROC AUC = {lr_roc:.4f}, AP = {lr_ap:.4f}, P@20 = {lr_p20:.4f}, P@50 = {lr_p50:.4f}")

# Train & Evaluate Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_tr[X_cols], y_tr)
dt_probs = dt.predict_proba(X_te[X_cols])[:, 1]
dt_roc = roc_auc_score(y_te, dt_probs)
dt_ap = average_precision_score(y_te, dt_probs)
dt_p20 = precision_at_k(y_te, dt_probs, 20)
dt_p50 = precision_at_k(y_te, dt_probs, 50)
print(f"Decision Tree: ROC AUC = {dt_roc:.4f}, AP = {dt_ap:.4f}, P@20 = {dt_p20:.4f}, P@50 = {dt_p50:.4f}")

# Train & Evaluate Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr[X_cols], y_tr)
rf_probs = rf.predict_proba(X_te[X_cols])[:, 1]
rf_roc = roc_auc_score(y_te, rf_probs)
rf_ap = average_precision_score(y_te, rf_probs)
rf_p20 = precision_at_k(y_te, rf_probs, 20)
rf_p50 = precision_at_k(y_te, rf_probs, 50)
print(f"Random Forest: ROC AUC = {rf_roc:.4f}, AP = {rf_ap:.4f}, P@20 = {rf_p20:.4f}, P@50 = {rf_p50:.4f}")


Loading dataset from the warehouse (mid-panel month partitions)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded clean dataset of 101,441 content items.
Train: 93,085 rows | Test: 8,356 rows | Overlapping clients: 0
Baseline: ROC AUC = 0.5476, AP = 0.5319, P@20 = 1.0000, P@50 = 0.9800
Logistic Regression: ROC AUC = 0.5757, AP = 0.5970, P@20 = 0.7000, P@50 = 0.7800
Decision Tree: ROC AUC = 0.5414, AP = 0.5124, P@20 = 0.3500, P@50 = 0.4400
Random Forest: ROC AUC = 0.5799, AP = 0.5356, P@20 = 0.5500, P@50 = 0.6200


## 4. Errors and interpretation

## 4. Errors and interpretation

### Feature Importances (Random Forest Model, client-grouped split)

- **`mar_avg_position` (30.13%):** still the single strongest driver — a page's search
  rank remains the clearest signal of where it's heading.
- **`mar_impressions` (28.23%):** visibility volume, close second.
- **`content_age_days` (24.36%):** staleness carries real weight even though the simple
  365-day bucket test in Week 4 showed almost no relationship — the model is finding a
  more continuous, non-linear use of age that a single threshold couldn't see.
- **`mar_clicks` (8.95%)** and **`mar_ga4_sessions_per_day` (8.34%):** supporting signals,
  each contributing under 10%.

### Analysis of concrete wrong predictions (hardest test-set errors)

1. **True label: declining (1) | Predicted prob: 0.0000** — Impressions 1,420, Clicks 4,
   Position 3.6, Sessions/day 1.33, Age 20 days. The model was maximally confident this page
   was *safe* — young, page-one position, real engagement — yet it declined anyway. This is
   the model's blind spot: a strong current snapshot doesn't rule out a decline driven by
   something outside these five features (e.g. a competitor change, a seasonal dip, a SERP
   feature shift) that isn't visible in March's numbers alone.

2. **True label: not declining (0) | Predicted prob: 1.0000** — Impressions 148, Clicks 0,
   Position 5.5, Sessions/day 0, Age 55 days. The model read "zero clicks, zero sessions" as
   a strong decline signal, but this page held steady anyway. A young, decent-position page
   with genuinely low but *stable* engagement isn't automatically at risk — zero engagement
   this month doesn't necessarily mean falling engagement next month.

3. **True label: declining (1) | Predicted prob: 0.0000** — Impressions 1,734, Clicks 22,
   Position 2.3, Sessions/day 2.1, Age 20 days. Nearly identical story to Case 1: a young,
   top-3-position, well-engaged page that the model was confident would hold — and it
   didn't.

**Pattern across the errors:** two of the three hardest mistakes are the model being
*overconfident that a young, well-ranked, engaged page is safe* — when it actually declined.
This points to a real limitation: the five available features describe a page's current
health well, but say little about *why* a healthy-looking page might still lose visibility
next month. That's a genuine gap for future feature work (e.g. content-type or
competitive-context signals), not something this feature set can fix.

In [7]:
# Code displaying top feature importances and extracting top error cases
import numpy as np

print("RF Feature Importances:")
for f, imp in zip(X_cols, rf.feature_importances_):
    print(f"  {f}: {imp:.4f}")

X_te_copy = X_te.copy()
X_te_copy['rf_prob'] = rf_probs
X_te_copy['error'] = np.abs(X_te_copy['is_declining'] - X_te_copy['rf_prob'])
most_wrong = X_te_copy.sort_values('error', ascending=False).head(3)

print("\nTop 3 Hardest Error Cases in Test Set:")
for idx, row in most_wrong.iterrows():
    print(f"- Client: {row['client_hash_id'][:8]}... | Content: {row['content_hash_id'][:8]}...")
    print(f"  True label: {row['is_declining']} | Predicted prob: {row['rf_prob']:.4f}")
    print(f"  Impressions: {row['mar_impressions']:.1f} | Clicks: {row['mar_clicks']:.1f} | Position: {row['mar_avg_position']:.1f}")
    print(f"  Sessions/day: {row['mar_ga4_sessions_per_day']:.4f} | Content Age Days: {row['content_age_days']:.1f}")


RF Feature Importances:
  mar_impressions: 0.2823
  mar_clicks: 0.0895
  mar_avg_position: 0.3013
  mar_ga4_sessions_per_day: 0.0834
  content_age_days: 0.2436

Top 3 Hardest Error Cases in Test Set:
- Client: client_0... | Content: content_...
  True label: 1 | Predicted prob: 0.0000
  Impressions: 1420.0 | Clicks: 4.0 | Position: 3.6
  Sessions/day: 1.3333 | Content Age Days: 20.0
- Client: client_2... | Content: content_...
  True label: 0 | Predicted prob: 1.0000
  Impressions: 148.0 | Clicks: 0.0 | Position: 5.5
  Sessions/day: 0.0000 | Content Age Days: 55.0
- Client: client_0... | Content: content_...
  True label: 1 | Predicted prob: 0.0000
  Impressions: 1734.0 | Clicks: 22.0 | Position: 2.3
  Sessions/day: 2.1000 | Content Age Days: 20.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.